In [3]:
!pip install nltk


In [5]:
import nltk
nltk.download("wordnet")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
import numpy as np
import pandas as pd
import re
import sklearn
from nltk.corpus import stopwords
from nltk import WordNetLemmatizer
lem = WordNetLemmatizer()

In [28]:
from google.colab import files
df = pd.read_csv('/content/all_kindle_review.csv')

In [9]:
df.head(2)

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400


In [29]:
# We only need the reviews and ratings columns
df = df [['reviewText','rating']]
df.head(2)

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5


In [30]:
# Checking if there are null values
df.isnull().sum()

,0
reviewText,0
rating,0


In [12]:
# Checking the number of unique values
df['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


In [31]:
# Reducing the target feature to only two classes
df['rating'] = df['rating'].apply(lambda x: 0 if x<3 else 1)
df['rating'].value_counts()

,count
rating,
1,8000
0,4000


In [32]:
# Preprocessing the text feature
df['reviewText'] = (
    df['reviewText']
    .astype(str)
    .str.strip()
    .str.replace(r'[^a-z A-z 0-9-]+', '', regex=True)
    .str.replace(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , regex=True)
    .str.lower()
)

In [15]:
df.head(2)

,reviewText,rating
0,jace rankin may be short but hes nothing to me...,1
1,great short read i didnt want to put it down ...,1


In [16]:
from bs4 import BeautifulSoup

In [33]:
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join([lem.lemmatize(y) for y in x.split() if y not in stopwords.words("english")]))
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x,'lxml').get_text())
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join(x.split()))

In [19]:
df.head(4)

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1


In [39]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20)

In [44]:
to_dense = FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)
pipeline = Pipeline(
    [
        ('tfidf',TfidfVectorizer(max_features=2000)),
        ('to_dense', to_dense),
        ('nb',GaussianNB())
    ]
)

param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2), (1,3)],
    'tfidf__max_df': [0.8, 0.9, 1.0],
    'tfidf__min_df': [1, 2, 5],
    #'nb__alpha': [0.01, 0.1, 1, 5]
}


grid = GridSearchCV(
    pipeline,
    cv = 5,
    n_jobs = -1,
    verbose = 0,
    param_grid= param_grid
)


In [ ]:
grid.fit(X_train,y_train)

In [46]:
y_pred = grid.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7620833333333333
              precision    recall  f1-score   support

           0       0.60      0.82      0.70       793
           1       0.89      0.73      0.80      1607

    accuracy                           0.76      2400
   macro avg       0.75      0.78      0.75      2400
weighted avg       0.80      0.76      0.77      2400

